# By-person analysis of the LoL cube rotos

Who drives lanes, who blends, who freelances — player-level views over
the three-draft package data. Vocabulary: a **Lane** is a card core
maindecked by three different people (one per draft) plus its flex
orbit; a **Team** is the 5+ card block two decks from different drafts
agreed on; a deck **merges** when it holds teams with two different
decks of the same other draft.

Regenerate data first if stale: `python3 refresh.py --dry-run`.

In [1]:
import contextlib, io, os, subprocess, sys
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from packages import (load, maindeck_owners, signature_groups, deck_sets,
                      team_partners, straddles, load_themes, load_scryfall,
                      card_colors, theme_str)

with contextlib.redirect_stderr(io.StringIO()):  # known junk-card warnings
    drafts, cube, decks = load()
owners = maindeck_owners(drafts, cube, decks)
groups = signature_groups(owners)
by_deck = deck_sets(owners)
partners = team_partners(owners)
merged = straddles(owners, drafts)
themes, scry = load_themes(), load_scryfall()
colors = {c: card_colors(c, scry) for c, _, _ in cube}
def colors_of(cards):
    u = set().union(*(colors[c] for c in cards)) if cards else set()
    return "".join(x for x in "WUBRG" if x in u) or "C"
lanes_of = {}
for gi, (sig, cards) in enumerate(groups):
    for k, p in enumerate(sig):
        lanes_of.setdefault((k, p), []).append(gi)
print(f"{len(drafts)} drafts, {sum(len(d.players) for d in drafts)} players, "
      f"{len(groups)} lanes, "
      f"{sum(len(pl) for bd in partners.values() for pl in bd.values())//2} team pairings")

3 drafts, 28 players, 11 lanes, 44 team pairings


## Per-player summary — one row per deck: what did each person build, own, and merge?

In [3]:
rows = []
for k, d in enumerate(drafts):
    for p in d.players:
        deck = by_deck[(k, p)]
        w, l = d.records.get(p, (0, 0))
        lanes = [f"P{gi+1} {theme_str(groups[gi][1], themes)}"
                 for gi in lanes_of.get((k, p), [])]
        n_teams = sum(len(pl) for pl in partners[(k, p)].values())
        n_merge = sum(1 for pl in merged.get((k, p), {}).values())
        rows.append({"draft": d.name, "player": p, "cards": len(deck),
                     "colors": colors_of(deck), "lanes": ", ".join(lanes) or "—",
                     "teams": n_teams, "merge cases": n_merge,
                     "record": f"{w}–{l}"})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

  draft           player  cards colors                       lanes  teams  merge cases record
Draft 1           Arason     25    WBR                           —      3            1    2–3
Draft 1     Stardust2187     34  WUBRG       P3 Ramp, P5 Graveyard      4            2    2–4
Draft 1             Mark     28    UBG                  P9 Discard      3            1    1–4
Draft 1          Raybees     26     WR                   P1 Tokens      2            0    2–4
Draft 1           Flooey     34    WUR        P2 Spells, P4 Spells      4            2    5–1
Draft 1     BladeTheKing     40   WUBG                           —      4            2    0–1
Draft 1 DefeatistElitist     30    WUB               P8 Rectangles      4            1    3–3
Draft 1       mattimedes     32  WUBRG      P6 Ramp, P11 Graveyard      5            2    0–0
Draft 1       Jack Heart     32    WBR P7 Sacrifice, P10 Sacrifice      4            1    5–0
Draft 2       imrahil327     30    WBG               P8 Rect

## Taxonomy — blenders, mergers, lane-drivers, free agents

In [5]:
all_decks = [(k, p) for k, d in enumerate(drafts) for p in d.players]
blend = [d for d, bd in merged.items() if len(bd) == 2]
single = [d for d, bd in merged.items() if len(bd) == 1]
non = [d for d in all_decks if d not in merged]
drivers = [d for d in non if d in lanes_of]
free = [d for d in non if d not in lanes_of]
for label, ds in [("Both-way blenders", blend), ("Single-side mergers", single),
                  ("Pure lane-drivers", drivers), ("Free agents", free)]:
    names = ", ".join(f"{drafts[k].name[-1]}:{p}" for k, p in sorted(ds))
    print(f"{label:20s} {len(ds):2d}  {names}")

Both-way blenders     6  1:BladeTheKing, 1:Flooey, 1:Stardust2187, 1:mattimedes, 2:lordtupperware, 3:ColdBrewNate
Single-side mergers  15  1:Arason, 1:DefeatistElitist, 1:Jack Heart, 1:Mark, 2:Aviseras, 2:Dawn, 2:aefunk, 2:imrahil327, 2:j_mazz_2020, 2:loosterbooster, 2:roc, 3:Greg, 3:Sefrox, 3:bengolds, 3:guilom
Pure lane-drivers     5  1:Raybees, 3:Balbadorf, 3:Maxim Sinistral, 3:Melissa, 3:llich
Free agents           2  2:tox 🍉, 3:FOOMP


## Multi-lane players — same theme twice (the seam inside one archetype) vs genuinely two archetypes

In [7]:
for (k, p), gis in sorted(lanes_of.items()):
    if len(gis) < 2:
        continue
    ts = [theme_str(groups[gi][1], themes) for gi in gis]
    kind = "SAME theme twice" if len(set(ts)) == 1 else "cross-theme"
    print(f"{drafts[k].name} {p:16s} {' + '.join(ts):24s} ({kind})")

Draft 1 Flooey           Spells + Spells          (SAME theme twice)
Draft 1 Jack Heart       Sacrifice + Sacrifice    (SAME theme twice)
Draft 1 Stardust2187     Ramp + Graveyard         (cross-theme)
Draft 1 mattimedes       Ramp + Graveyard         (cross-theme)
Draft 2 Aviseras         Tokens + Sacrifice       (cross-theme)
Draft 2 aefunk           Graveyard + Graveyard    (SAME theme twice)
Draft 2 j_mazz_2020      Spells + Spells          (SAME theme twice)
Draft 2 lordtupperware   Ramp + Ramp              (SAME theme twice)
Draft 3 Greg             Ramp + Ramp              (SAME theme twice)
Draft 3 guilom           Graveyard + Graveyard    (SAME theme twice)


## Merge detail — who merged what, against whom

In [9]:
for (k, p), bd in sorted(merged.items()):
    for k2, plist in sorted(bd.items()):
        parts = ", ".join(
            f"{pb} [{len(core)}"
            + (f" {theme_str(core, themes)}" if theme_str(core, themes) else "")
            + "]"
            for pb, core in sorted(plist, key=lambda x: -len(x[1])))
        print(f"{drafts[k].name} {p:16s} vs {drafts[k2].name}: {parts}")

Draft 1 Arason           vs Draft 2: Dawn [8], Aviseras [5]
Draft 1 BladeTheKing     vs Draft 2: lordtupperware [8], roc [6]
Draft 1 BladeTheKing     vs Draft 3: ColdBrewNate [11], bengolds [6]
Draft 1 DefeatistElitist vs Draft 2: lordtupperware [8], imrahil327 [6 Rectangles], roc [5]
Draft 1 Flooey           vs Draft 2: j_mazz_2020 [17 Spells], loosterbooster [5]
Draft 1 Flooey           vs Draft 3: llich [12 Spells], ColdBrewNate [8 Spells]
Draft 1 Jack Heart       vs Draft 2: Aviseras [7 Sacrifice], Dawn [6 Sacrifice], imrahil327 [5]
Draft 1 Mark             vs Draft 3: Greg [5], Sefrox [5 Discard]
Draft 1 Stardust2187     vs Draft 2: aefunk [8 Graveyard], loosterbooster [5]
Draft 1 Stardust2187     vs Draft 3: Greg [13 Ramp], guilom [8 Graveyard]
Draft 1 mattimedes       vs Draft 2: lordtupperware [7 Ramp], aefunk [6 Graveyard]
Draft 1 mattimedes       vs Draft 3: FOOMP [9], Greg [8 Ramp], guilom [5 Graveyard]
Draft 2 Aviseras         vs Draft 1: Raybees [12 Tokens], Jack Heart [7 

## The merge report — full decklists with team groupings, as HTML

In [11]:
out = subprocess.run([sys.executable, "report.py"], cwd="..",
                     capture_output=True, text=True)
path = os.path.abspath(os.path.join("..", "out", "merge-report.html"))
print(f"written: {path}")
try:
    from IPython.display import IFrame, display
    display(IFrame(os.path.relpath(path), width="100%", height=600))
except ImportError:
    print("(open in a browser)")

written: /Users/jack/src/cube/.claude/worktrees/roto-min-package-3/roto/out/merge-report.html
(open in a browser)
